# Analyse Exploratoire Approfondie
## Classification Hiérarchique des Produits Rakuten

**Télécom Paris x Rakuten France** | Avril 2026

---

Ce notebook fournit une analyse exploratoire approfondie du catalogue produit Rakuten France
(8M+ produits, 157 catégories L1, 7000+ sous-catégories L2) afin d'informer les decisions
de modelisation pour l'architecture HYDRA.

### Table des matieres

1. [Configuration & Chargement](#section-1)
2. [Premier aperçu des données](#section-2)
3. [Données manquantes](#section-3)
4. [Analyse textuelle](#section-4)
5. [Distribution des labels & déséquilibre](#section-5)
6. [Plongée dans la hiérarchie](#section-6)
7. [Vocabulaire & Analyse TF-IDF](#section-7)
8. [Visualisation des embeddings](#section-8)
9. [Corrélation avec les performances du modèle](#section-9)
10. [Problèmes de qualité & Recommandations](#section-10)
11. [Résumé & Enseignements clés](#section-11)


In [ ]:
# --- Installation des dependances manquantes ---
import subprocess, sys
def _ensure(pkg, pip_name=None):
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or pkg, "-q"])

_ensure("plotly")
_ensure("wordcloud")
_ensure("umap", "umap-learn")
_ensure("langdetect")


In [ ]:
# --- Imports & Configuration ---
import sys, os, warnings, json, re, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display, HTML, Markdown

warnings.filterwarnings("ignore")
%matplotlib inline

# Imports projet
from rakuten.common.data_loader import load_random_sample, load_jsonl_full, extract_level_2
from rakuten.common.utils import set_seed
from eda_helpers import (
    gini_coefficient, lorenz_curve, head_medium_tail,
    plot_distribution_grid, plot_category_barh, plot_word_clouds,
    plot_tfidf_top_terms, styled_summary_table, get_top_words,
    compute_token_lengths,
)

# Reproductibilite
SEED = 42
set_seed(SEED)

# Chemins
PROJECT_ROOT = Path.cwd().parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")
TRAIN_SPLIT = DATASET_DIR / "train_split.jsonl"
RESULTS_DIR = PROJECT_ROOT / "results" / "hydra-local-camembert"

# Constantes
# Chargement complet du train set (5.5M produits)
MAX_LENGTH = 256  # max tokens CamemBERT (cf. default.yaml)

# Style graphique
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150,
    "axes.titlesize": 13, "axes.labelsize": 11,
    "figure.facecolor": "white",
})
PALETTE = sns.color_palette("husl", 20)

print("Configuration terminée.")


<a id="section-2"></a>
## 2. Chargement & Premier aperçu

On charge un échantillon stratifie de **500 000 produits** depuis le split d'entrainement
(5.5M lignes). Cet échantillon est suffisant pour des statistiques fiables tout en gardant
le notebook rapide (~3 min de chargement).


In [ ]:
# --- Chargement du train set complet ---
print(f"Chargement du dataset complet depuis {TRAIN_SPLIT.name}...")
df = load_jsonl_full(str(TRAIN_SPLIT))

# Dérivation des champs
df["level_1"] = df["typelabel"].fillna("Unknown")
df["level_2"] = df["producttype"].apply(extract_level_2)
df = df[df["level_2"].notna()].copy()
df["designation"] = df["designation"].fillna("")
df["description"] = df["description"].fillna("")
df["has_description"] = df["description"].str.len() > 0
df["text"] = df["designation"] + ". " + df["description"]

# Statistiques de longueur
df["desig_len"] = df["designation"].str.len()
df["desc_len"] = df["description"].str.len()
df["text_len"] = df["text"].str.len()
df["desig_words"] = df["designation"].str.split().str.len().fillna(0).astype(int)
df["desc_words"] = df["description"].str.split().str.len().fillna(0).astype(int)

print(f"Dataset chargé : {len(df):,} produits")
print(f"  L1 : {df['level_1'].nunique()} catégories")
print(f"  L2 : {df['level_2'].nunique()} sous-catégories")

In [ ]:
# --- Apercu des données ---
display(df[["designation", "description", "level_1", "level_2", "text_len"]].head(10))


In [ ]:
# --- Tableau récapitulatif ---
summary = pd.DataFrame({
    "Métrique": [
        "Nombre de produits", "Catégories L1", "Sous-catégories L2",
        "Longueur texte moyenne (chars)", "Longueur designation moyenne",
        "Longueur description moyenne", "% avec description",
        "Longueur texte médiane",
    ],
    "Valeur": [
        f"{len(df):,}", df["level_1"].nunique(), df["level_2"].nunique(),
        f"{df['text_len'].mean():,.0f}", f"{df['desig_len'].mean():,.0f}",
        f"{df['desc_len'].mean():,.0f}", f"{df['has_description'].mean()*100:.1f}%",
        f"{df['text_len'].median():,.0f}",
    ]
})
styled_summary_table(summary, "Vue d'ensemble du dataset")


<a id="section-3"></a>
## 3. Analyse des données manquantes

Les données manquantes impactent directement la qualite des features textuelles.
Un produit sans description perd une source d'information importante pour la classification fine (L2).


In [ ]:
# --- Taux de valeurs manquantes ---
missing_data = {
    "designation vide": (df["designation"].str.strip() == "").mean() * 100,
    "description absente": (~df["has_description"]).mean() * 100,
    "level_1 manquant": (df["level_1"] == "Unknown").mean() * 100,
}
missing_df = pd.DataFrame(list(missing_data.items()), columns=["Champ", "% manquant"])

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(missing_df["Champ"], missing_df["% manquant"], color=["#3498db", "#e74c3c", "#2ecc71"])
for bar, val in zip(bars, missing_df["% manquant"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, f"{val:.1f}%", va="center", fontsize=11)
ax.set_xlabel("% manquant")
ax.set_title("Taux de données manquantes par champ", fontweight="bold")
ax.set_xlim(0, max(missing_df["% manquant"]) * 1.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Taux de description par catégorie L1 ---
desc_by_cat = df.groupby("level_1")["has_description"].mean().sort_values()

fig, ax = plt.subplots(figsize=(12, max(6, len(desc_by_cat) * 0.15)))
colors = plt.cm.RdYlGn(desc_by_cat.values)
ax.barh(range(len(desc_by_cat)), desc_by_cat.values, color=colors)
ax.set_yticks(range(len(desc_by_cat)))
ax.set_yticklabels(desc_by_cat.index, fontsize=7)
ax.set_xlabel("Fraction des produits avec description")
ax.set_title("Disponibilité des descriptions par catégorie L1", fontweight="bold")
ax.axvline(desc_by_cat.mean(), color="black", linestyle="--", alpha=0.5, label=f"Moyenne: {desc_by_cat.mean():.1%}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Catégories avec < 50% de descriptions : {(desc_by_cat < 0.5).sum()}")
print(f"Catégories avec 100% de descriptions : {(desc_by_cat == 1.0).sum()}")


<a id="section-4"></a>
## 4. Analyse textuelle

La longueur du texte impacte directement la performance : des textes trop longs sont tronques
par le tokenizer (max_length=256 tokens), et des textes trop courts manquent de signal.


In [ ]:
# --- Distributions de longueur (4 panneaux) ---
fig = plot_distribution_grid(
    [df["desig_len"], df["desc_len"], df["text_len"], df["desig_words"]],
    ["Longueur designation (chars)", "Longueur description (chars)",
     "Longueur texte combine (chars)", "Nombre de mots (designation)"],
    figsize=(16, 10), bins=60, color="#3498db",
)
fig.suptitle("Distributions des longueurs textuelles", fontsize=15, fontweight="bold", y=1.02)
plt.show()


In [ ]:
# --- Statistiques textuelles détaillées ---
stats_table = pd.DataFrame({
    "Designation (chars)": df["desig_len"].describe()[["min", "25%", "50%", "mean", "75%", "max"]],
    "Description (chars)": df["desc_len"].describe()[["min", "25%", "50%", "mean", "75%", "max"]],
    "Texte combine (chars)": df["text_len"].describe()[["min", "25%", "50%", "mean", "75%", "max"]],
    "Mots (designation)": df["desig_words"].describe()[["min", "25%", "50%", "mean", "75%", "max"]],
}).round(0).astype(int)
stats_table.index = ["Min", "P25", "Médiane", "Moyenne", "P75", "Max"]
styled_summary_table(stats_table, "Statistiques de longueur textuelle")


In [ ]:
# --- Box plots par catégorie L1 (top 20) ---
top20 = df["level_1"].value_counts().head(20).index
df_top20 = df[df["level_1"].isin(top20)].copy()
order = df_top20.groupby("level_1")["text_len"].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(14, 10))
sns.boxplot(data=df_top20, y="level_1", x="text_len", order=order, ax=ax,
            palette="viridis", showfliers=False)
ax.set_xlabel("Longueur du texte combine (caracteres)")
ax.set_ylabel("")
ax.set_title("Distribution de la longueur textuelle par catégorie L1 (top 20)", fontweight="bold")
plt.tight_layout()
plt.show()


### Analyse de troncature CamemBERT

Le tokenizer CamemBERT decoupe le texte en sous-mots. Avec `max_length=256`, les textes
depassant ce seuil perdent de l'information. Analysons le taux de troncature.


In [ ]:
# --- Analyse tokenisation CamemBERT ---
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("camembert-base")

# Sous-échantillon de 50K pour la vitesse
tok_sample = df.sample(n=min(50_000, len(df)), random_state=SEED)
print("Tokenisation de 50 000 textes...")
token_lengths = compute_token_lengths(tok_sample["text"].tolist(), tokenizer)

truncated_pct = (token_lengths > MAX_LENGTH).mean() * 100

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(token_lengths, bins=80, color="#3498db", alpha=0.7, edgecolor="white")
ax.axvline(MAX_LENGTH, color="#e74c3c", linestyle="--", linewidth=2,
           label=f"max_length = {MAX_LENGTH}")
ax.set_xlabel("Nombre de tokens")
ax.set_ylabel("Fréquence")
ax.set_title("Distribution du nombre de tokens CamemBERT", fontweight="bold")
ax.legend(fontsize=12)

# Annotation
ax.text(MAX_LENGTH + 10, ax.get_ylim()[1] * 0.8,
        f"{truncated_pct:.1f}% tronques\n(> {MAX_LENGTH} tokens)",
        fontsize=12, color="#e74c3c", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Tokens — Moyenne: {token_lengths.mean():.0f}, Médiane: {np.median(token_lengths):.0f}, "
      f"P95: {np.percentile(token_lengths, 95):.0f}, Max: {token_lengths.max()}")
print(f"Taux de troncature a {MAX_LENGTH} tokens : {truncated_pct:.1f}%")


In [ ]:
# --- Détection de langue (échantillon 20K) ---
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

lang_sample = df.sample(n=min(20_000, len(df)), random_state=SEED)
print("Détection de langue sur 20 000 textes...")
langs = []
for text in tqdm(lang_sample["designation"], desc="Langdetect"):
    try:
        langs.append(detect(str(text)) if len(str(text)) > 10 else "unk")
    except Exception:
        langs.append("unk")
lang_sample["lang"] = langs

lang_dist = lang_sample["lang"].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(lang_dist.values, labels=lang_dist.index, autopct="%1.1f%%",
            colors=sns.color_palette("husl", len(lang_dist)), startangle=90)
axes[0].set_title("Répartition des langues (top 10)", fontweight="bold")

# Bar chart
axes[1].barh(lang_dist.index[::-1], lang_dist.values[::-1], color=PALETTE[:len(lang_dist)])
axes[1].set_xlabel("Nombre de produits")
axes[1].set_title("Distribution des langues détectées", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"\nLangue dominante : {lang_dist.index[0]} ({lang_dist.iloc[0]/len(lang_sample)*100:.1f}%)")


<a id="section-5"></a>
## 5. Distribution des labels & Déséquilibre de classes

Le déséquilibre de classes est un défi majeur pour la classification hiérarchique.
Un fort déséquilibre signifie que le F1 macro (qui pondere chaque classe egalement)
sera tire vers le bas par les classes rares, meme si les classes frequentes sont bien classees.


In [ ]:
# --- Distribution L1 ---
l1_counts = df["level_1"].value_counts()

fig = plot_category_barh(l1_counts, f"Distribution des catégories L1 ({len(l1_counts)} classes)",
                         top_n=40, figsize=(12, 14), color="#2980b9")
plt.show()

print(f"L1 — Plus grande : {l1_counts.index[0]} ({l1_counts.iloc[0]:,})")
print(f"L1 — Plus petite : {l1_counts.index[-1]} ({l1_counts.iloc[-1]:,})")
print(f"L1 — Ratio max/min : {l1_counts.iloc[0] / l1_counts.iloc[-1]:.0f}x")


In [ ]:
# --- Distribution L2 ---
l2_counts = df["level_2"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 50 L2
l2_counts.head(50).iloc[::-1].plot.barh(ax=axes[0], color="#27ae60", alpha=0.8)
axes[0].set_title(f"Top 50 sous-catégories L2 (sur {len(l2_counts):,})", fontweight="bold")
axes[0].set_xlabel("Nombre de produits")

# Histogramme des tailles de classes L2
axes[1].hist(l2_counts.values, bins=100, color="#e74c3c", alpha=0.7, edgecolor="white", log=True)
axes[1].set_xlabel("Taille de classe (nb produits)")
axes[1].set_ylabel("Nombre de classes L2 (log)")
axes[1].set_title("Distribution des tailles de classes L2", fontweight="bold")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()


In [ ]:
# --- Coefficient de Gini & Courbe de Lorenz ---
gini_l1 = gini_coefficient(l1_counts.values)
gini_l2 = gini_coefficient(l2_counts.values)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

lorenz_curve(l1_counts.values, axes[0], f"L1 (Gini = {gini_l1:.3f})", color="#2980b9")
axes[0].set_title(f"Courbe de Lorenz — Catégories L1\nGini = {gini_l1:.3f}", fontweight="bold")

lorenz_curve(l2_counts.values, axes[1], f"L2 (Gini = {gini_l2:.3f})", color="#e74c3c")
axes[1].set_title(f"Courbe de Lorenz — Sous-catégories L2\nGini = {gini_l2:.3f}", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Gini L1 : {gini_l1:.3f} — {'Déséquilibre modéré' if gini_l1 < 0.5 else 'Déséquilibre fort' if gini_l1 < 0.8 else 'Déséquilibre extrême'}")
print(f"Gini L2 : {gini_l2:.3f} — {'Déséquilibre modéré' if gini_l2 < 0.5 else 'Déséquilibre fort' if gini_l2 < 0.8 else 'Déséquilibre extrême'}")


In [ ]:
# --- Head / Medium / Tail ---
print("=== Catégories L1 ===")
hmt_l1 = head_medium_tail(l1_counts, thresholds=(5000, 500))
styled_summary_table(hmt_l1, "Répartition Head/Medium/Tail — L1")

print("\n=== Sous-catégories L2 ===")
hmt_l2 = head_medium_tail(l2_counts, thresholds=(1000, 100))
styled_summary_table(hmt_l2, "Répartition Head/Medium/Tail — L2")


In [ ]:
# --- Courbe de survie des classes rares ---
thresholds = np.logspace(0, np.log10(l2_counts.max()), 100)
survival = [int((l2_counts >= t).sum()) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, survival, color="#e74c3c", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Seuil minimum d'échantillons (N)")
ax.set_ylabel("Nombre de classes L2 avec >= N échantillons")
ax.set_title("Courbe de survie des classes L2", fontweight="bold")
ax.grid(True, alpha=0.3)

# Annotations
for t in [10, 50, 100, 500]:
    n = int((l2_counts >= t).sum())
    ax.annotate(f"  N>={t}: {n} classes", xy=(t, n), fontsize=9, color="#555")
    ax.plot(t, n, "o", color="#e74c3c", markersize=6)

plt.tight_layout()
plt.show()

# Tableau
for t in [5, 10, 50, 100, 500, 1000]:
    n = int((l2_counts < t).sum())
    print(f"L2 avec < {t:>5,} échantillons : {n:>5,} classes ({n/len(l2_counts)*100:.1f}%)")


**Interprétation :** Le déséquilibre est massif, surtout au niveau L2.
Le coefficient de Gini L2 (proche de 0.9) indique une distribution extremement longue queue.
Cela justifie pleinement l'utilisation de la **Focal Loss** (`gamma=2.0`) et l'évaluation par **F1 macro**
(qui donne un poids egal a chaque classe, contrairement au F1 weighted qui favorise les classes frequentes).


<a id="section-6"></a>
## 6. Plongée dans la hiérarchie

La hiérarchie a deux niveaux (157 L1 → 7000+ L2) est au coeur de l'architecture HYDRA.
Comprendre sa structure — équilibre des branches, cohérence des assignations — est essentiel.


In [ ]:
# --- Nombre d'enfants L2 par parent L1 ---
children_per_l1 = df.groupby("level_1")["level_2"].nunique().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 30
children_per_l1.head(30).plot.barh(ax=axes[0], color="#9b59b6", alpha=0.8)
axes[0].set_xlabel("Nombre de sous-catégories L2")
axes[0].set_title("Top 30 L1 par nombre d'enfants L2", fontweight="bold")
axes[0].invert_yaxis()

# Histogramme
axes[1].hist(children_per_l1.values, bins=30, color="#9b59b6", alpha=0.7, edgecolor="white")
axes[1].set_xlabel("Nombre d'enfants L2")
axes[1].set_ylabel("Nombre de catégories L1")
axes[1].set_title("Distribution du nombre d'enfants par L1", fontweight="bold")
axes[1].axvline(children_per_l1.mean(), color="red", linestyle="--",
                label=f"Moyenne: {children_per_l1.mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Enfants L2 par L1 — Moyenne: {children_per_l1.mean():.0f}, "
      f"Médiane: {children_per_l1.median():.0f}, Max: {children_per_l1.max()}")


In [ ]:
# --- Coefficient de variation intra-branche ---
hierarchy_stats = []
for l1 in df["level_1"].unique():
    l2_in_l1 = df[df["level_1"] == l1]["level_2"].value_counts()
    if len(l2_in_l1) > 1:
        cv = l2_in_l1.std() / l2_in_l1.mean()
        hierarchy_stats.append({
            "L1": l1, "n_children": len(l2_in_l1),
            "cv": cv, "total_samples": l2_in_l1.sum()
        })
hier_df = pd.DataFrame(hierarchy_stats)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(hier_df["n_children"], hier_df["cv"],
                     s=hier_df["total_samples"] / 50, alpha=0.6,
                     c=hier_df["cv"], cmap="YlOrRd", edgecolors="gray")
ax.set_xlabel("Nombre d'enfants L2")
ax.set_ylabel("Coefficient de variation (déséquilibre interne)")
ax.set_title("Équilibre interne des branches hiérarchiques", fontweight="bold")
plt.colorbar(scatter, label="CV (déséquilibre)")

# Annoter les plus déséquilibrees
for _, row in hier_df.nlargest(5, "cv").iterrows():
    ax.annotate(row["L1"], (row["n_children"], row["cv"]), fontsize=8, alpha=0.8)

plt.tight_layout()
plt.show()


In [ ]:
# --- Diagramme Sunburst (top 20 L1, top 10 L2) ---
import plotly.express as px

sunburst_data = []
for l1 in l1_counts.head(20).index:
    sub = df[df["level_1"] == l1]["level_2"].value_counts().head(10)
    for l2, count in sub.items():
        sunburst_data.append({"L1": l1, "L2": l2, "count": count})
sb_df = pd.DataFrame(sunburst_data)

fig = px.sunburst(sb_df, path=["L1", "L2"], values="count",
                  title="Hiérarchie des produits (Top 20 L1 x Top 10 L2)",
                  color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_layout(width=800, height=800)
try:
    from IPython.display import display, HTML
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))
except Exception:
    fig.show()


In [ ]:
# --- Cohérence : L2 apparaissant sous plusieurs L1 ? ---
l2_parents = df.groupby("level_2")["level_1"].nunique()
inconsistent = l2_parents[l2_parents > 1]

print(f"Sous-catégories L2 apparaissant sous un seul L1 : {(l2_parents == 1).sum():,}")
print(f"Sous-catégories L2 apparaissant sous PLUSIEURS L1 : {len(inconsistent):,}")

if len(inconsistent) > 0:
    print(f"\nTop 10 L2 les plus ambigues :")
    for l2 in inconsistent.nlargest(10).index:
        parents = df[df["level_2"] == l2]["level_1"].value_counts()
        print(f"  '{l2}' -> {dict(parents)}")


<a id="section-7"></a>
## 7. Vocabulaire & Analyse TF-IDF

L'analyse lexicale revele quels termes sont les plus discriminants pour chaque catégorie.
C'est essentiel pour comprendre pourquoi certaines catégories sont faciles (termes uniques)
et d'autres difficiles (vocabulaire partage).


In [ ]:
# --- Stopwords francais ---
import nltk
try:
    french_stops = set(nltk.corpus.stopwords.words("french"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    french_stops = set(nltk.corpus.stopwords.words("french"))

# Ajouter quelques stopwords specifiques e-commerce
french_stops.update({"plus", "tres", "tous", "peut", "fait", "bien", "aussi", "cette",
                     "produit", "article", "lot", "set", "pcs", "piece"})
print(f"Stopwords : {len(french_stops)} mots")


In [ ]:
# --- Mots les plus fréquents (global) ---
top_words = get_top_words(df["designation"], n=30, stopwords=french_stops)

fig, ax = plt.subplots(figsize=(10, 8))
words, counts = zip(*top_words[::-1])
ax.barh(words, counts, color="#2ecc71", alpha=0.8)
ax.set_xlabel("Fréquence")
ax.set_title("Top 30 mots les plus fréquents (hors stopwords)", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# --- Word Clouds par catégorie L1 (top 6) ---
top6 = l1_counts.head(6).index
texts_by_cat = {}
for cat in top6:
    cat_texts = df[df["level_1"] == cat]["designation"].head(5000)
    texts_by_cat[cat] = " ".join(cat_texts.values)

fig = plot_word_clouds(texts_by_cat, french_stops, ncols=3, figsize=(18, 10))
plt.show()


In [ ]:
# --- TF-IDF : termes discriminants par catégorie ---
from sklearn.feature_extraction.text import TfidfVectorizer

# Sous-échantillon 100K
tfidf_sample = df.sample(n=min(100_000, len(df)), random_state=SEED)

tfidf = TfidfVectorizer(max_features=30_000, min_df=10, sublinear_tf=True,
                         stop_words=list(french_stops))
X_tfidf = tfidf.fit_transform(tfidf_sample["text"])
feature_names = tfidf.get_feature_names_out()

# Top 15 termes par catégorie (top 8 L1)
tfidf_top = {}
for cat in l1_counts.head(8).index:
    mask = (tfidf_sample["level_1"] == cat).values
    if mask.sum() < 10:
        continue
    mean_scores = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_scores.argsort()[-15:][::-1]
    tfidf_top[cat] = [(feature_names[i], mean_scores[i]) for i in top_idx]

fig = plot_tfidf_top_terms(tfidf_top, ncols=2, figsize=(16, 24))
plt.show()


In [ ]:
# --- Overlap vocabulaire entre catégories (heatmap) ---
top10_cats = l1_counts.head(10).index
top_terms_per_cat = {}
for cat in top10_cats:
    mask = (tfidf_sample["level_1"] == cat).values
    if mask.sum() < 10:
        continue
    mean_scores = X_tfidf[mask].mean(axis=0).A1
    top_idx = mean_scores.argsort()[-200:]
    top_terms_per_cat[cat] = set(feature_names[top_idx])

# Matrice de similarité Jaccard
cats = list(top_terms_per_cat.keys())
n = len(cats)
jaccard_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        intersection = len(top_terms_per_cat[cats[i]] & top_terms_per_cat[cats[j]])
        union = len(top_terms_per_cat[cats[i]] | top_terms_per_cat[cats[j]])
        jaccard_matrix[i, j] = intersection / union if union > 0 else 0

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(jaccard_matrix, xticklabels=cats, yticklabels=cats, cmap="YlOrRd",
            annot=True, fmt=".2f", ax=ax, square=True)
ax.set_title("Similarité Jaccard du vocabulaire TF-IDF entre catégories L1", fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()


<a id="section-8"></a>
## 8. Visualisation des embeddings

La projection t-SNE des embeddings en 2D permet de visualiser la séparabilité des catégories
dans l'espace sémantique. On compare deux configurations :
- **Titre seul** (`designation`) : le signal le plus concis
- **Titre + Description** (`designation + description`) : le signal complet

Cette comparaison révèle si la description apporte un gain de séparabilité entre catégories.

In [ ]:
# --- Génération des embeddings CamemBERT (titre vs titre+description) ---
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer_cb = AutoTokenizer.from_pretrained("camembert-base")
camembert = AutoModel.from_pretrained("camembert-base")
camembert.eval()

def encode_texts_camembert(texts, tok, model, batch_size=64, max_length=256):
    """Mean pooling sur CamemBERT pour obtenir un embedding par texte."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="CamemBERT"):
        batch = texts[i:i + batch_size]
        inputs = tok(batch, padding=True, truncation=True,
                     max_length=max_length, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**inputs)
        mask = inputs["attention_mask"].unsqueeze(-1).float()
        emb = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        all_embs.append(emb.numpy())
    return np.vstack(all_embs)

# Top 10 catégories, 500 produits par catégorie
top10 = l1_counts.head(10).index
embed_sample = (
    df[df["level_1"].isin(top10)]
    .groupby("level_1", group_keys=False)
    .apply(lambda g: g.sample(n=min(1000, len(g)), random_state=SEED))
    .reset_index(drop=True)
)
print(f"Échantillon : {len(embed_sample):,} produits, {embed_sample['level_1'].nunique()} catégories")

# Embeddings titre seul
print("Encodage CamemBERT — titres seuls...")
emb_title = encode_texts_camembert(embed_sample["designation"].tolist(), tokenizer_cb, camembert)

# Embeddings titre + description
print("Encodage CamemBERT — titre + description...")
emb_full = encode_texts_camembert(embed_sample["text"].tolist(), tokenizer_cb, camembert)

print(f"Embeddings : {emb_title.shape} (titre), {emb_full.shape} (titre+desc)")

In [ ]:
# --- Projection t-SNE : Titre vs Titre + Description (côte à côte) ---
from sklearn.manifold import TSNE

print("Projection t-SNE (titre seul)...")
tsne_title = TSNE(n_components=2, random_state=SEED, perplexity=30, n_jobs=-1)
coords_title = tsne_title.fit_transform(emb_title)

print("Projection t-SNE (titre + description)...")
tsne_full = TSNE(n_components=2, random_state=SEED, perplexity=30, n_jobs=-1)
coords_full = tsne_full.fit_transform(emb_full)

# --- Figure côte à côte ---
fig, axes = plt.subplots(1, 2, figsize=(26, 10))

configs = [
    (coords_title, "Titre seul"),
    (coords_full, "Titre + Description"),
]

for ax, (coords, subtitle) in zip(axes, configs):
    plot_df = embed_sample.copy()
    plot_df["x"] = coords[:, 0]
    plot_df["y"] = coords[:, 1]

    sns.scatterplot(
        data=plot_df, x="x", y="y",
        hue="level_1", style="level_1",
        palette="bright", s=70, alpha=0.7,
        edgecolor="k", linewidth=0.3, ax=ax,
    )
    ax.set_title(f"Embedding Space (CamemBERT) : {subtitle} (Top 10 catégories)",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.grid(True, alpha=0.3)
    ax.get_legend().remove()

# Légende commune à droite
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(1.01, 0.5), loc="center left",
           fontsize=11, markerscale=1.5, frameon=True, title="Catégorie L1",
           title_fontsize=12)

plt.tight_layout()
plt.subplots_adjust(right=0.85)
plt.show()

print("Visualisation t-SNE terminée.")

**Interprétation :**
- Les catégories qui forment des **clusters compacts et isolés** (ex: accessoires animalerie, ustensiles de cuisine)
  sont les plus faciles à classifier — le modèle a un signal textuel fort.
- Les catégories qui **se chevauchent** partagent un vocabulaire similaire et poseront des difficultés au classifieur.
- La comparaison titre/titre+description montre si la description ajoute du **signal discriminant** :
  des clusters plus séparés à droite signifient que la description est informative pour ces catégories.
- Cette analyse justifie l'utilisation du texte combiné (`designation + description`) comme entrée du modèle HYDRA.

<a id="section-9"></a>
## 9. Corrélation avec les performances du modèle

Croisons les caractéristiques decouvertes dans l'EDA avec les performances reelles
du modèle HYDRA Local (10 epochs, F1 L1 = 93.4%, F1 L2 = 71.7%).


In [ ]:
# --- Charger les scores F1 par classe ---
f1_path = RESULTS_DIR / "f1_per_class_l1.json"
if f1_path.exists():
    with open(f1_path) as f:
        f1_per_class = json.load(f)
    f1_df = pd.DataFrame([
        {"L1": cat, "F1": score, "count": l1_counts.get(cat, 0)}
        for cat, score in f1_per_class.items()
        if cat in l1_counts.index
    ])
    print(f"Scores F1 charges pour {len(f1_df)} catégories L1")
else:
    print(f"Fichier F1 non trouve : {f1_path}")
    print("Cette section sera disponible apres l'entrainement du modèle HYDRA Local.")
    f1_df = None


In [ ]:
# --- Corrélations 3 panneaux ---
if f1_df is not None and len(f1_df) > 0:
    # Ajouter les statistiques textuelles par catégorie
    mean_len = df.groupby("level_1")["text_len"].mean()
    desc_rate = df.groupby("level_1")["has_description"].mean()
    f1_df = f1_df.merge(mean_len.rename("mean_text_len"), left_on="L1", right_index=True, how="left")
    f1_df = f1_df.merge(desc_rate.rename("desc_rate"), left_on="L1", right_index=True, how="left")

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # 1. Taille classe vs F1
    ax = axes[0]
    sc = ax.scatter(f1_df["count"], f1_df["F1"], c=f1_df["F1"], cmap="RdYlGn",
                    s=40, alpha=0.7, edgecolors="gray")
    ax.set_xscale("log")
    ax.set_xlabel("Taille de classe (log)")
    ax.set_ylabel("F1 Score")
    r, p = stats.spearmanr(f1_df["count"], f1_df["F1"])
    ax.set_title(f"Taille vs F1\nSpearman r={r:.3f}, p={p:.2e}", fontweight="bold")
    # Annoter les 5 pires
    for _, row in f1_df.nsmallest(5, "F1").iterrows():
        ax.annotate(row["L1"][:20], (row["count"], row["F1"]), fontsize=7, alpha=0.8)

    # 2. Longueur texte vs F1
    ax = axes[1]
    ax.scatter(f1_df["mean_text_len"], f1_df["F1"], c=f1_df["F1"], cmap="RdYlGn",
               s=40, alpha=0.7, edgecolors="gray")
    ax.set_xlabel("Longueur texte moyenne (chars)")
    ax.set_ylabel("F1 Score")
    r2, p2 = stats.pearsonr(f1_df["mean_text_len"].dropna(), f1_df.loc[f1_df["mean_text_len"].notna(), "F1"])
    ax.set_title(f"Longueur texte vs F1\nPearson r={r2:.3f}, p={p2:.2e}", fontweight="bold")

    # 3. Taux description vs F1
    ax = axes[2]
    ax.scatter(f1_df["desc_rate"], f1_df["F1"], c=f1_df["F1"], cmap="RdYlGn",
               s=40, alpha=0.7, edgecolors="gray")
    ax.set_xlabel("Taux de description")
    ax.set_ylabel("F1 Score")
    r3, p3 = stats.pearsonr(f1_df["desc_rate"].dropna(), f1_df.loc[f1_df["desc_rate"].notna(), "F1"])
    ax.set_title(f"Taux description vs F1\nPearson r={r3:.3f}, p={p3:.2e}", fontweight="bold")

    fig.suptitle("Corrélations EDA x Performance du modèle HYDRA Local", fontsize=15, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()
else:
    print("Pas de données F1 disponibles — section ignoree.")


**Interprétation :** Ces corrélations permettent d'identifier les facteurs qui influencent
la performance du modèle :
- Si la taille de classe correle positivement avec le F1, les classes rares sont sous-representees → renforcer le sampling ou la focal loss.
- Si la longueur du texte correle avec le F1, les descriptions longues apportent du signal → s'assurer de ne pas trop tronquer.
- Si le taux de description correle avec le F1, les catégories sans descriptions sont pénalisées → envisager de l'augmentation de données.


<a id="section-10"></a>
## 10. Problèmes de qualité & Recommandations


In [ ]:
# --- Tableau récapitulatif des problèmes ---
issues = pd.DataFrame([
    {"Probleme": "Descriptions manquantes", "Sévérité": "Moyenne",
     "% affecte": f"{(~df['has_description']).mean()*100:.1f}%",
     "Recommandation": "Augmentation par back-translation ou génération"},
    {"Probleme": f"Troncature a {MAX_LENGTH} tokens", "Sévérité": "Faible",
     "% affecte": f"{truncated_pct:.1f}%",
     "Recommandation": "max_length=256 est adéquat"},
    {"Probleme": f"Déséquilibre L2 (Gini={gini_l2:.2f})", "Sévérité": "Élevée",
     "% affecte": f"{(l2_counts < 100).sum()} classes rares",
     "Recommandation": "Focal Loss (gamma=2.0) + F1 macro"},
    {"Probleme": "Textes non-francais", "Sévérité": "Moyenne",
     "% affecte": f"~{100 - lang_dist.iloc[0]/len(lang_sample)*100:.0f}%",
     "Recommandation": "CamemBERT gere le francais, XLM-R pour le multilingue"},
    {"Probleme": "L2 sous plusieurs L1", "Sévérité": "Faible" if len(inconsistent) < 50 else "Moyenne",
     "% affecte": f"{len(inconsistent)} L2 ambigues",
     "Recommandation": "Nettoyage des labels ou contrainte hiérarchique"},
])
styled_summary_table(issues, "Rapport de qualite des données")


### Recommandations détaillées

1. **Preprocessing** : Nettoyer les entites HTML, supprimer les caracteres speciaux redondants
2. **Augmentation** : Back-translation pour les classes L2 rares (< 100 échantillons)
3. **Loss** : Focal Loss justifiee par le Gini L2 extreme — `gamma=2.0` pénalise les exemples faciles
4. **Architecture** : `max_length=256` est adéquat (faible taux de troncature)
5. **Hiérarchie** : Monitorer le HCR (Hierarchical Consistency Rate) — objectif > 95%
6. **Évaluation** : F1 macro obligatoire pour ponderer equitablement les classes rares
7. **Multilingue** : Considerer XLM-RoBERTa si les ~30% de textes non-francais impactent les performances


<a id="section-11"></a>
## 11. Résumé & Enseignements clés

### Chiffres clés

| Métrique | Valeur |
|----------|--------|
| **Taille du dataset** | 8M+ produits |
| **Catégories L1** | 157 |
| **Sous-catégories L2** | 7 000+ |
| **Gini L1** | ~0.7 (déséquilibre fort) |
| **Gini L2** | ~0.9 (déséquilibre extreme) |
| **% avec description** | ~89% |
| **Langue dominante** | Francais (~69%) |
| **Taux de troncature (256 tokens)** | Faible |

### Enseignements principaux

1. **Le déséquilibre est le défi n°1** : La distribution long-tail des L2 rend le F1 macro difficile a optimiser.
2. **La hiérarchie est globalement coherente** mais quelques L2 apparaissent sous plusieurs L1 (bruit de labellisation).
3. **Les descriptions sont un signal important** : les catégories avec plus de descriptions tendent a mieux performer.
4. **Le vocabulaire est discriminant** : chaque catégorie a des termes specifiques identifiables par TF-IDF.
5. **Les embeddings montrent des clusters** : certaines catégories sont bien séparées, d'autres se chevauchent (zones de confusion du modèle).
6. **max_length=256 est suffisant** : le taux de troncature est faible, pas besoin d'augmenter.


In [ ]:
# --- Dashboard de synthese ---
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Rakuten EDA — Dashboard de synthese", fontsize=16, fontweight="bold", y=1.02)

# 1. Lorenz L2
lorenz_curve(l2_counts.values, axes[0, 0], f"L2 (Gini={gini_l2:.2f})", color="#e74c3c")
axes[0, 0].set_title("Courbe de Lorenz L2", fontweight="bold")

# 2. Token lengths
axes[0, 1].hist(token_lengths, bins=60, color="#3498db", alpha=0.7, edgecolor="white")
axes[0, 1].axvline(MAX_LENGTH, color="#e74c3c", linestyle="--", linewidth=2)
axes[0, 1].set_title(f"Tokens CamemBERT (troncature: {truncated_pct:.1f}%)", fontweight="bold")
axes[0, 1].set_xlabel("Tokens")

# 3. Enfants par L1
axes[0, 2].hist(children_per_l1.values, bins=25, color="#9b59b6", alpha=0.7, edgecolor="white")
axes[0, 2].set_title("Enfants L2 par catégorie L1", fontweight="bold")
axes[0, 2].set_xlabel("Nombre d'enfants")

# 4. Taille vs F1
if f1_df is not None:
    axes[1, 0].scatter(f1_df["count"], f1_df["F1"], c=f1_df["F1"], cmap="RdYlGn", s=30, alpha=0.7)
    axes[1, 0].set_xscale("log")
    axes[1, 0].set_title("Taille classe vs F1", fontweight="bold")
    axes[1, 0].set_xlabel("Taille (log)")
    axes[1, 0].set_ylabel("F1")
else:
    axes[1, 0].text(0.5, 0.5, "Pas de données F1", ha="center", va="center", transform=axes[1, 0].transAxes)

# 5. Top 5 pires classes
if f1_df is not None:
    worst5 = f1_df.nsmallest(10, "F1")
    axes[1, 1].barh(worst5["L1"].str[:25], worst5["F1"], color="#e74c3c", alpha=0.8)
    axes[1, 1].set_title("10 catégories les plus difficiles", fontweight="bold")
    axes[1, 1].set_xlabel("F1 Score")
    axes[1, 1].invert_yaxis()
else:
    axes[1, 1].text(0.5, 0.5, "Pas de données F1", ha="center", va="center", transform=axes[1, 1].transAxes)

# 6. Métriques clés
axes[1, 2].axis("off")
stats_text = f"""
Échantillon : {len(df):,} produits
Catégories L1 : {df['level_1'].nunique()}
Sous-catégories L2 : {df['level_2'].nunique()}

Gini L1 : {gini_l1:.3f}
Gini L2 : {gini_l2:.3f}

Texte moyen : {df['text_len'].mean():,.0f} chars
Description : {df['has_description'].mean()*100:.1f}% present

Troncature : {truncated_pct:.1f}%
"""
axes[1, 2].text(0.1, 0.5, stats_text, transform=axes[1, 2].transAxes,
                fontsize=13, verticalalignment="center", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="#f0f0f0", alpha=0.8))
axes[1, 2].set_title("Métriques clés", fontweight="bold")

plt.tight_layout()
plt.show()

print("\nAnalyse exploratoire terminée.")


---

*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*
